# cpt_jyc_2026_stock_bar1m 数据质量检查

逐交易日检查 2020 至 2024 年的数据：

1. 每只股票每天是否有 250 分钟数据；
2. 与 `cn_stock_bar1m_derived_c` 内连接后，覆盖率和公共字段匹配率是否达标；
3. 只输出未通过的日期和股票数量，同时写入 `test.log`。

In [ ]:
from pathlib import Path

import dai
import numpy as np
import pandas as pd

START_DATE = "2020-01-01"
END_DATE = "2024-12-31"
DATASOURCE = "cpt_jyc_2026_stock_bar1m"
REFERENCE_DATASOURCE = "cn_stock_bar1m_derived_c"
LOG_FILE = Path.cwd() / "test.log"

EXPECTED_MINUTES = 250
MIN_JOIN_RATE = 0.95
MIN_FIELD_MATCH_RATE = 0.95
JOIN_KEYS = ["date", "instrument"]

In [ ]:
def query(sql, day):
    return dai.query(
        sql,
        filters={"date": [f"{day} 00:00:00", f"{day} 23:59:59"]},
        compression=True,
    ).df()


def get_trading_days():
    df = dai.query(
        """
        SELECT DISTINCT date
        FROM cn_stock_index_component
        WHERE instrument = '000852.SH'
        """,
        filters={"date": [START_DATE, END_DATE]},
    ).df()
    return sorted(pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d").unique())


def compare_values(left, right):
    if pd.api.types.is_numeric_dtype(left) and pd.api.types.is_numeric_dtype(right):
        return np.isclose(
            pd.to_numeric(left, errors="coerce"),
            pd.to_numeric(right, errors="coerce"),
            rtol=1e-4,
            atol=1e-4,
            equal_nan=True,
        )
    return left.eq(right) | (left.isna() & right.isna())

In [ ]:
def get_failed_instruments(actual, reference):
    actual = actual.copy()
    reference = reference.copy()
    actual["date"] = pd.to_datetime(actual["date"])
    reference["date"] = pd.to_datetime(reference["date"])

    minute_counts = actual.groupby("instrument")["date"].nunique()
    minute_failed = set(minute_counts[minute_counts != EXPECTED_MINUTES].index)

    fields = sorted(set(actual.columns) & set(reference.columns) - set(JOIN_KEYS))
    merged = actual[JOIN_KEYS + fields].merge(
        reference[JOIN_KEYS + fields],
        on=JOIN_KEYS,
        how="inner",
        suffixes=("_actual", "_reference"),
    )

    joined_counts = merged.groupby("instrument").size()
    join_rates = joined_counts.div(minute_counts).fillna(0)
    join_failed = set(join_rates[join_rates < MIN_JOIN_RATE].index)
    join_failed.update(set(minute_counts.index) - set(joined_counts.index))

    if merged.empty or not fields:
        field_failed = set(minute_counts.index)
    else:
        matches = pd.DataFrame({
            field: compare_values(
                merged[f"{field}_actual"], merged[f"{field}_reference"]
            )
            for field in fields
        })
        matches["instrument"] = merged["instrument"].values
        match_rates = matches.groupby("instrument")[fields].mean().mean(axis=1)
        field_failed = set(match_rates[match_rates < MIN_FIELD_MATCH_RATE].index)

    return minute_failed, join_failed, field_failed

In [ ]:
trading_days = get_trading_days()
failed_days = 0

with LOG_FILE.open("w", encoding="utf-8") as log_file:
    def log(message):
        print(message)
        print(message, file=log_file, flush=True)

    for day in trading_days:
        actual = query(f"SELECT * FROM {DATASOURCE}", day)
        reference = query(f"SELECT * FROM {REFERENCE_DATASOURCE}", day)

        if actual.empty:
            log(f"{day}: 目标数据为空")
            failed_days += 1
            continue

        minute_failed, join_failed, field_failed = get_failed_instruments(
            actual, reference
        )
        failed = minute_failed | join_failed | field_failed
        if failed:
            failed_days += 1
            log(
                f"{day}: 未通过 {len(failed)} 只股票 "
                f"(分钟数 {len(minute_failed)}, "
                f"内连接 {len(join_failed)}, 字段差异 {len(field_failed)})"
            )

    log(f"检查完成：共 {len(trading_days)} 个交易日，{failed_days} 个日期未通过")
    log(f"日志文件：{LOG_FILE.resolve()}")